In [1]:
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer

In [2]:
from pathlib import Path
import pandas as pd

# Locate project directory
BASE_DIR = Path.cwd().parent

# Load cleaned dataset
data_path = BASE_DIR / "data" / "processed" / "cleaned_support_tickets.csv"

df = pd.read_csv(data_path)

print("Total tickets:", len(df))
df.head()

Total tickets: 20


,id,subject,body,correct_reply,category
0,FC001,Damaged organic apples,"I received my organic apples today, but severa...",We apologize for the condition of your apples....,Organic Produce Quality
1,FC002,Wilted spinach received,The spinach in my grocery order looks wilted a...,We are sorry that the spinach did not meet you...,Organic Produce Quality
2,FC003,Mold on strawberries,I opened my strawberry package and noticed mol...,We apologize for this experience. Please do no...,Organic Produce Quality
3,FC004,Incorrect organic vegetables,I ordered organic carrots but received regular...,We are sorry for the incorrect item. Please sh...,Organic Produce Quality
4,FC005,Delivery is late,"My grocery delivery was scheduled for 2 PM, bu...",We apologize for the delay. Please share your ...,Delivery Delays


In [3]:
df["combined_text"] = (
    df["subject"].fillna("").str.strip() + ". " +
    df["body"].fillna("").str.strip()
)

df[["id", "combined_text"]].head()

,id,combined_text
0,FC001,Damaged organic apples. I received my organic ...
1,FC002,Wilted spinach received. The spinach in my gro...
2,FC003,Mold on strawberries. I opened my strawberry p...
3,FC004,Incorrect organic vegetables. I ordered organi...
4,FC005,Delivery is late. My grocery delivery was sche...


In [4]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!


In [5]:
# Convert combined customer messages into a list
sentences = df["combined_text"].tolist()

# Generate embeddings
embeddings = model.encode(
    sentences,
    show_progress_bar=True
)

print("Embeddings generated successfully!")
print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings generated successfully!
Embedding shape: (20, 384)


In [6]:
print("Number of tickets:", len(sentences))
print("Embedding shape:", embeddings.shape)
print("First embedding (first 10 values):", embeddings[0][:10])

Number of tickets: 20
Embedding shape: (20, 384)
First embedding (first 10 values): [ 0.05256562  0.04971226  0.08964946  0.00103613  0.06124413 -0.05693919
  0.013894    0.04059675  0.00269173  0.01085067]


In [7]:
%pip install faiss-cpu

   ---------------------------------------- 0.0/16.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.6 MB ? eta -:--:--
    --------------------------------------- 0.3/16.6 MB ? eta -:--:--
    --------------------------------------- 0.3/16.6 MB ? eta -:--:--
   - -------------------------------------- 0.5/16.6 MB 540.5 kB/s eta 0:00:30
   - -------------------------------------- 0.5/16.6 MB 540.5 kB/s eta 0:00:30
   - -------------------------------------- 0.5/16.6 MB 540.5 kB/s eta 0:00:30
   - -------------------------------------- 0.8/16.6 MB 486.4 kB/s eta 0:00:33
   - -------------------------------------- 0.8/16.6 MB 486.4 kB/s eta 0:00:33
   - -------------------------------------- 0.8/16.6 MB 486.4 kB/s eta 0:00:33
   - -------------------------------------- 0.8/16.6 MB 486.4 kB/s eta 0:00:33
   - ----------------------


[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import faiss
import numpy as np

print("FAISS version:", faiss.__version__)
print("NumPy version:", np.__version__)

FAISS version: 1.16.0
NumPy version: 2.4.6


In [9]:
# Convert embeddings into NumPy float32 format
embedding_vectors = np.array(embeddings).astype("float32")

print("Shape:", embedding_vectors.shape)
print("Data type:", embedding_vectors.dtype)

Shape: (20, 384)
Data type: float32


In [10]:
# Get the number of dimensions in each embedding
dimension = embedding_vectors.shape[1]

# Create a FAISS index using L2 distance
index = faiss.IndexFlatL2(dimension)

# Add all ticket embeddings to the index
index.add(embedding_vectors)

print("FAISS index created successfully!")
print("Total vectors stored:", index.ntotal)
print("Vector dimensions:", index.d)

FAISS index created successfully!
Total vectors stored: 20
Vector dimensions: 384


In [11]:
# New customer query
query = "My strawberries arrived rotten and damaged."

# Convert query into an embedding
query_embedding = model.encode([query])

# Convert it into float32
query_vector = np.array(query_embedding).astype("float32")

# Search for the 3 most similar tickets
distances, indices = index.search(query_vector, k=3)

print("Retrieved ticket indices:", indices)
print("Similarity distances:", distances)

Retrieved ticket indices: [[2 0 1]]
Similarity distances: [[0.69586074 0.88323724 1.1331372 ]]


In [12]:
# Retrieve the matching tickets from our DataFrame
results = df.iloc[indices[0]][
    ["id", "subject", "body", "category", "correct_reply"]
]

results

,id,subject,body,category,correct_reply
2,FC003,Mold on strawberries,I opened my strawberry package and noticed mol...,Organic Produce Quality,We apologize for this experience. Please do no...
0,FC001,Damaged organic apples,"I received my organic apples today, but severa...",Organic Produce Quality,We apologize for the condition of your apples....
1,FC002,Wilted spinach received,The spinach in my grocery order looks wilted a...,Organic Produce Quality,We are sorry that the spinach did not meet you...


In [13]:
# Define the vector store directory
vector_store_dir = BASE_DIR / "vector_store"

# Save the FAISS index
faiss.write_index(
    index,
    str(vector_store_dir / "faiss.index")
)

print("FAISS index saved successfully!")

FAISS index saved successfully!


In [14]:
import json

# Select the information we need to retrieve later
metadata = df[
    ["id", "subject", "body", "category", "correct_reply"]
].to_dict(orient="records")

# Save metadata as JSON
metadata_path = vector_store_dir / "metadata.json"

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=4, ensure_ascii=False)

print("Metadata saved successfully!")
print("Total records:", len(metadata))
print("Saved at:", metadata_path)

Metadata saved successfully!
Total records: 20
Saved at: d:\Ezitech\freshcart-rag-knowledge-base\vector_store\metadata.json


In [15]:
# Check whether both files exist
index_path = vector_store_dir / "faiss.index"
metadata_path = vector_store_dir / "metadata.json"

print("FAISS index exists:", index_path.exists())
print("Metadata exists:", metadata_path.exists())

# Check their file sizes
print("FAISS index size:", index_path.stat().st_size, "bytes")
print("Metadata size:", metadata_path.stat().st_size, "bytes")

FAISS index exists: True
Metadata exists: True
FAISS index size: 30765 bytes
Metadata size: 8657 bytes


In [16]:
# Load the saved FAISS index
loaded_index = faiss.read_index(str(index_path))

# Load the saved metadata
with open(metadata_path, "r", encoding="utf-8") as file:
    loaded_metadata = json.load(file)

print("Index loaded successfully!")
print("Vectors:", loaded_index.ntotal)
print("Dimensions:", loaded_index.d)

print("\nMetadata loaded successfully!")
print("Records:", len(loaded_metadata))

Index loaded successfully!
Vectors: 20
Dimensions: 384

Metadata loaded successfully!
Records: 20


In [17]:
# Search using the loaded FAISS index
distances, indices = loaded_index.search(
    query_vector,
    k=3
)

# Retrieve matching records from saved metadata
for i, idx in enumerate(indices[0]):
    ticket = loaded_metadata[idx]

    print(f"\nResult {i + 1}")
    print("Ticket ID:", ticket["id"])
    print("Subject:", ticket["subject"])
    print("Category:", ticket["category"])
    print("Distance:", distances[0][i])
    print("Correct reply:", ticket["correct_reply"])


Result 1
Ticket ID: FC003
Subject: Mold on strawberries
Category: Organic Produce Quality
Distance: 0.69586074
Correct reply: We apologize for this experience. Please do not consume the affected strawberries. Share a photo and your order number with our support team so we can review the issue and arrange an appropriate resolution.

Result 2
Ticket ID: FC001
Subject: Damaged organic apples
Category: Organic Produce Quality
Distance: 0.88323724
Correct reply: We apologize for the condition of your apples. Please share a photo of the damaged items and your order number with our support team. We will review your request and assist with an eligible refund or replacement.

Result 3
Ticket ID: FC002
Subject: Wilted spinach received
Category: Organic Produce Quality
Distance: 1.1331372
Correct reply: We are sorry that the spinach did not meet your expectations. Please provide your order number and a photo of the product so our team can review the quality issue and guide you through the availa